In [1]:
#@title Install Dependencies and Download Models

%pip install -q transformers datasets

import torch
from transformers import GPT2Tokenizer, GPT2LMHeadModel
from transformers import LogitsProcessor, set_seed
import numpy as np
import datasets

tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
model = GPT2LMHeadModel.from_pretrained("gpt2")

Note: you may need to restart the kernel to use updated packages.


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

In [2]:
if torch.backends.mps.is_available():
    device = 'mps'
elif torch.cuda.is_available():
    device = 'cuda:0'
else:
    device = 'cpu'

model.to(device)
model = model.eval()
print(f'Using device: {device}')

Using device: mps


In [3]:
# @title Download lab files

import sys

!rm -rf llm_lab

![ ! -d 'llm_lab' ] && git clone https://github.com/ethz-privsec/llm_lab.git
%cd llm_lab
!git pull https://github.com/ethz-privsec/llm_lab.git
%cd ..
if "llm_lab" not in sys.path:
  sys.path.append("llm_lab")

Cloning into 'llm_lab'...
remote: Enumerating objects: 152, done.
remote: Counting objects: 100% (152/152), done.
remote: Compressing objects: 100% (102/102), done.
remote: Total 152 (delta 80), reused 114 (delta 48), pack-reused 0 (from 0)
Receiving objects: 100% (152/152), 503.66 KiB | 7.63 MiB/s, done.
Resolving deltas: 100% (80/80), done.
/Users/gorkemkadirsolun/Library/CloudStorage/GoogleDrive-gorkemkadirsolun@gmail.com/My Drive/Ders_Course/Large_Language_Models/large-language-models-homeworks/third/question1/llm_lab
From https://github.com/ethz-privsec/llm_lab
 * branch            HEAD       -> FETCH_HEAD
Already up to date.
/Users/gorkemkadirsolun/Library/CloudStorage/GoogleDrive-gorkemkadirsolun@gmail.com/My Drive/Ders_Course/Large_Language_Models/large-language-models-homeworks/third/question1


In [4]:
# @title Example of how to generate 100 tokens of text without watermarking

from llm_lab.gpt_generate import generate_with_seed, gen_red_list

prompt = "Boston is one of the oldest municipalities in America,"
print(generate_with_seed(model, tokenizer, prompt, seed=42))

[transformers] The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Boston is one of the oldest municipalities in America, and its name will get much more iconic as it makes a rise into broadcast television. It's home to some 1 million children from all over Florida who are now expected by their parents or guardians — but many here can't afford an adult schedule because they live with kids for decades compared on-screen family members (or siblings).
— Andrew Maltz ·· 20 years ago 0 Thumbs up 3Th thumbs down Report Abuse | Site Error / News Filter Livejournal Legal Help: Contact Us By clicking


You will now implement three different watermarking schemes:
1. A simple scheme that never outputs the letter 'e' (lowercase or uppercase)
2. A red-list scheme, that generates a random list of banned tokens for each token generation.
3. A soft red-list scheme, that also generates a random red-list, but just biases the LLM against these tokens instead of outright banning them, by substracting the value `logit_offset=2` from the logits of each red-listed token.

You should implement each of these schemes as a `LogitsProcessor` class.

For the red-list schemes, you should use `gpt_generate.gen_red_list` to generate a red list containing 50% of the LLM's vocabulary.
The seed for generating the pseudorandom red list is computed from the previous token processed by the model.

So for example, if the model has so far processed the string "my name is " (which tokenizes as `[1820, 1438, 318, 220]`), then the red list for the next token to be generated is `**gen_red_list(torch.LongTensor([220]), model.config.vocab_size)** = [43383,  7006, 40846, ...]`.

In [5]:
#@title Exercise 1: Implement a trivial watermarking scheme that samples text without any 'e' (lowercase or uppercase)

class NoEsLogitsProcessor(LogitsProcessor):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.e_token_ids = [
            token_id
            for token_id in range(tokenizer.vocab_size)
            if "e" in tokenizer.decode([token_id]).lower()
        ]

    def __call__(self, input_ids, scores):
        """
        Processes the output scores of the LLM before generating the next token.
        Args:
            input_ids: torch.LongTensor of shape (batch_size, sequence_length) — Indices of input sequence tokens in the vocabulary.
            scores: torch.FloatTensor of shape (batch_size, model.config.vocab_size) — Logits for the next token to be generated.
        Returns: torch.FloatTensor of shape (batch_size, model.config.vocab_size) — The processed logits.
        """
        scores[:, self.e_token_ids] = -float("inf")
        return scores


no_e_processor = NoEsLogitsProcessor()

prompt = "Anton Vowl is missing. Ransacking his Paris flat, a group of his faithful companions trawl through his diary for any hint as to his location and, insidiously, a ghost, from Vowl's past starts to cast its malignant shadow.\n "
output = generate_with_seed(model, tokenizer, prompt, logits_processor=no_e_processor, seed=42)
print(output)

Anton Vowl is missing. Ransacking his Paris flat, a group of his faithful companions trawl through his diary for any hint as to his location and, insidiously, a ghost, from Vowl's past starts to cast its malignant shadow.
  Yuri Volkman – In physics class I taught my T-shirt company about fabricating an optical illusion using 'paracord' (a sort that adds colour too) which would allow us full visual motion without thinking: it was going in with this twist on paracords! As now our blimp pulls up into front again at 1000mph or so but all flicks roll around looking silly if not downright ugly - no airy flying bird will do anything similar anyway ... Or


In [6]:
#@title Exercise 2: Implement a red-list watermarking scheme

class RedListLogitsProcessor(LogitsProcessor):
    def __init__(self, red_frac=0.5, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.red_frac = red_frac

    def __call__(self, input_ids, scores):
        """
        Processes the output scores of the LLM before generating the next token.
        Args:
            input_ids: torch.LongTensor of shape (batch_size, sequence_length) — Indices of input sequence tokens in the vocabulary.
            scores: torch.FloatTensor of shape (batch_size, model.config.vocab_size) — Logits for the next token to be generated.
        Returns: torch.FloatTensor of shape (batch_size, model.config.vocab_size) — The processed logits.
        """
        vocab_size = scores.shape[-1]
        for batch_idx in range(input_ids.shape[0]):
            prev_token_id = int(input_ids[batch_idx, -1].item())
            red_list = gen_red_list(prev_token_id, vocab_size, frac_red=self.red_frac)
            red_list = torch.as_tensor(red_list, dtype=torch.long, device=scores.device)
            scores[batch_idx, red_list] = -float("inf")
        return scores

red_list_processor = RedListLogitsProcessor()

prompt = "Boston is one of the oldest municipalities in America,"
output = generate_with_seed(model, tokenizer, prompt, logits_processor=red_list_processor, seed=42)
print(output)

Boston is one of the oldest municipalities in America, but its history and culture has never really caught on with people here. It didn't until I tried it a little bit earlier this year when free-speech groups were sitting down for dinner to write about issues like equal rights."


But she says these are just some things that change everyday within city governments because "every decision should be made from inside her or herself" – not as an elected member's office holder (though there may exist exceptions), which can lead someone into missteps at council meetings


In [7]:
#@title Exercise 3: Implement a soft red-list watermarking scheme

class SoftRedListLogitsProcessor(LogitsProcessor):
    def __init__(self, red_frac=0.5, logit_offset=2.0, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.red_frac = red_frac
        self.logit_offset = logit_offset

    def __call__(self, input_ids, scores):
        """
        Processes the output scores of the LLM before generating the next token.
        Args:
            input_ids: torch.LongTensor of shape (batch_size, sequence_length) — Indices of input sequence tokens in the vocabulary.
            scores: torch.FloatTensor of shape (batch_size, model.config.vocab_size) — Logits for the next token to be generated.
        Returns: torch.FloatTensor of shape (batch_size, model.config.vocab_size) — The processed logits.
        """
        vocab_size = scores.shape[-1]
        for batch_idx in range(input_ids.shape[0]):
            prev_token_id = int(input_ids[batch_idx, -1].item())
            red_list = gen_red_list(prev_token_id, vocab_size, frac_red=self.red_frac)
            red_list = torch.as_tensor(red_list, dtype=torch.long, device=scores.device)
            scores[batch_idx, red_list] -= self.logit_offset
        return scores

soft_red_list_processor = SoftRedListLogitsProcessor()

prompt = "Boston is one of the oldest municipalities in America,"
output = generate_with_seed(model, tokenizer, prompt, logits_processor=soft_red_list_processor, seed=42)
print(output)

Boston is one of the oldest municipalities in America, and its name will get a lot more special thanks to Samby Roy Collins. The former Chief Police Officer was awarded his own burial certificate by free-standing Santa Barbara County Sheriff Joe Lombardo when he presented it at an auction back then on Jan 30th with $100 million compared according
"It really fits that," says Steve Hanblom's brother Drew Marris . "I have just come here from Kansas City where I had been coming all summer for my family."  He points out there


Okay! We're now ready to start generating watermarked text.
We give you 20 prompts in `data/watermark_prompts.txt`.
For each of these, generate 100 more tokens using each of the three watermarking schemes, and save all of this as a numpy array for submission.

MAKE SURE TO USE `seed=42` FOR ALL YOUR GENERATIONS.

In [8]:
from tqdm import trange

with open('llm_lab/data/watermark_prompts.txt') as f:
  prompts = f.read().splitlines()

processors = [no_e_processor, red_list_processor, soft_red_list_processor]
outputs = []

seed = 42  # DON'T CHANGE THIS!!!

for i in trange(20):
  min_new_tokens = 100
  max_new_tokens = min_new_tokens

  for j in range(3):
    output = generate_with_seed(model, tokenizer, prompts[i], logits_processor=processors[j],
                                min_new_tokens=min_new_tokens, max_new_tokens=max_new_tokens, seed=seed)
    outputs.append(output)

print(outputs)

100%|██████████| 20/20 [00:56<00:00,  2.84s/it]

['The only way to deal with an unfree world is to become so absolutely free that your very existence is an act of rebellion. And you cannot possibly abolish capitalism – but at no point can anything truly stop it, until its own opposition stands against both \'Marxism\' and socialism!\nFor a good start I think Marxists such as Bakunin may do just about what most anarchists would call "anti-war". But in my opinion this approach simply won\'t work if our social institutions don�t allow us (just look how much war has cost Islamic radicals) - or any capitalist nation which did not attack Islam almost 100', 'The only way to deal with an unfree world is to become so absolutely free that your very existence is an act of rebellion. If we accept the view here, however minimal it may sound at first glance then I think there can still remain little else in us than this idea: We must subjugation." [23]\nAnd such are some practical ways for communicating about a certain kind and level which they wi

For the final part of this question, you now have to try and detect watermarked text.
We give you 80 pieces of text in `data/watermarked_gens.npy`.
For each piece of text you have to guess whether it was generated with:

1.   No watermark
2.   The dummy "no E's" watermark
3.   The red-list watermark
4.   The soft red-list watermark

We use the same `generate_with_seed` and `gen_red_list` implementations as you. Our red-list watermarking scheme also uses the same parameters (i.e., 50% of the tokens are red-listed, and for the soft version we substract 2.0 from the logits).

Exactly 20 of the 80 texts are generated with each of the 4 options above. Each text is comprised of a short prompt, followed by 100-200 generated tokens.

Store your guesses (1,2,3,4) for each piece of text in a numpy array.

In [9]:
from collections import defaultdict

outputs_secret = np.load("llm_lab/data/watermarked_gens.npy", allow_pickle=True)
assert len(outputs_secret) == 80

def red_list_rate(text, tokenizer, vocab_size, red_frac=0.5):
    token_ids = tokenizer.encode(str(text))
    if len(token_ids) < 2:
        return 0.0

    followers_by_prev = defaultdict(list)
    for prev_token, next_token in zip(token_ids[:-1], token_ids[1:]):
        followers_by_prev[int(prev_token)].append(int(next_token))

    red_hits = 0
    for prev_token, next_tokens in followers_by_prev.items():
        red_tokens = set(map(int, gen_red_list(prev_token, vocab_size, frac_red=red_frac)))
        red_hits += sum(next_token in red_tokens for next_token in next_tokens)

    return red_hits / (len(token_ids) - 1)

# The four classes are balanced. The dummy watermark is isolated by its tiny
# number of e/E characters, then the red-list schemes are ranked by red-hit rate.
e_counts = np.asarray([str(text).lower().count("e") for text in outputs_secret])
red_rates = np.asarray([
    red_list_rate(text, tokenizer, tokenizer.vocab_size)
    for text in outputs_secret
])

my_guesses = np.zeros(len(outputs_secret), dtype=int)

no_e_indices = np.argsort(e_counts)[:20]
my_guesses[no_e_indices] = 2

remaining = np.asarray([i for i in range(len(outputs_secret)) if my_guesses[i] == 0])
remaining_by_red_rate = remaining[np.argsort(red_rates[remaining])]

my_guesses[remaining_by_red_rate[:20]] = 3
my_guesses[remaining_by_red_rate[20:40]] = 4
my_guesses[remaining_by_red_rate[40:]] = 1

print({label: int(np.sum(my_guesses == label)) for label in [1, 2, 3, 4]})

{1: 20, 2: 20, 3: 20, 4: 20}


## Export your solution

To save your results, you can use the code below, which will save the file in Colab's temporary storage (or locally, if you're not using Colab), or on your Google Drive. If you save it on Colab's temporary storage, you can download it from there (see the file system icon on the left).

In [21]:
from llm_lab.utils import get_solution_path, is_valid_student_id
from pathlib import Path

#@markdown Check this box if you want to save your results on Google Drive. Otherwise they'll be
#@markdown saved on the ephimeral Colab storage. The storage will be deleted with the runtime,
#@markdown so REMEMBER TO DOWNLOAD THE FILES before you close the tab!
SAVE_ON_DRIVE = False # @param {"type":"boolean"}

#@markdown The number on your Legi (Student ID card). It's in the format 'dd-ddd-ddd'
STUDENT_ID = "25-936-154"  # @param {"type":"string","placeholder":"00-000-000"}

assert is_valid_student_id(STUDENT_ID), "Student ID should have the format 'dd-ddd-ddd'"

base_solutions_path = get_solution_path(STUDENT_ID, SAVE_ON_DRIVE)
SOLUTIONS_PATH = base_solutions_path.with_name(base_solutions_path.name + "_mps")
SOLUTIONS_PATH.mkdir(parents=True, exist_ok=True)

In [22]:
# Save generations
assert len(outputs) == 60
np.save(SOLUTIONS_PATH / "Q1_gens.npy", np.asarray(outputs))

# Save guesses
assert len(my_guesses) == 80
np.save(SOLUTIONS_PATH / "Q1_guesses.npy", np.asarray(my_guesses))